<a href="https://www.kaggle.com/code/saharjafari/flight-fare-price-prediction?scriptVersionId=236488971" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# **Flight Price Prediction Project**

This project demonstrates a comprehensive data analysis and machine learning pipeline aimed at predicting flight ticket prices based on various features such as duration, departure times, stops, and airlines. Below is a breakdown of the steps, from data preprocessing to feature engineering and exploratory data analysis (EDA), using Python libraries like Pandas, NumPy, Matplotlib, Seaborn, and Scikit-learn.

**Project Overview**

**Objective:**

Predict flight ticket prices using historical flight data through various machine learning models after thorough data cleaning, feature extraction, and visualization.

**Dataset:**

The dataset contains information about flights, including departure and arrival times, airline names, duration, number of stops, and price. The data is read from an Excel file using Pandas.

# **Table of Contents**

1. [Data Loading & Inspection](#data-loading--inspection)
2. [Data Cleaning & Feature Engineering](#data-cleaning--feature-engineering)
3. [Exploratory Data Analysis (EDA)](#exploratory-data-analysis-eda)
4. [Feature Encoding](#feature-encoding)
5. [Outlier Detection and Treatment](#outlier-detection-and-treatment)
6. [Feature Selection Using Mutual Information](#feature-selection-using-mutual-information)
7. [Model Building](#model-building)
8. [Model Saving and Deployment](#model-saving-and-deployment)
9. [Hyperparameter Tuning](#hyperparameter-tuning)
10. [Conclusion](#conclusion)


## **1. Data Loading & Inspection**

In [ ]:
# Importing necessary libraries for data manipulation, visualization, and modeling
import numpy as np  # For numerical operations
import pandas as pd  # For data manipulation
import matplotlib.pyplot as plt  # For plotting visualizations
import seaborn as sns  # For advanced visualizations

# Reading the dataset from an Excel file
dataset = pd.read_excel('/kaggle/input/flight-price/Data_Train.xlsx')

# Display the first few rows of the dataset to understand its structure
dataset.head()

# Checking the structure of the dataset to understand column types and missing values
dataset.info()

# Identifying the total number of missing values in each column
dataset.isnull().sum()

# Dropping rows with missing values to simplify preprocessing
# In real scenarios, other strategies like imputation could be considered
dataset.dropna(inplace=True)


## **2. Data Cleaning & Feature Engineering**

**Date & Time Feature Transformation:**

In [ ]:
# Function to convert columns from string format to datetime objects
def change_into_datetime(col):
    dataset[col] = pd.to_datetime(dataset[col], format=None)

# Converting departure time, arrival time, and journey date columns to datetime objects
for feature in ['Dep_Time', 'Arrival_Time', 'Date_of_Journey']:
    change_into_datetime(feature)

# Extracting day, month, and year from the Date_of_Journey column for additional insights
dataset['Journey_Day'] = dataset['Date_of_Journey'].dt.day
dataset['Journey_Month'] = dataset['Date_of_Journey'].dt.month
dataset['Journey_Year'] = dataset['Date_of_Journey'].dt.year


**Extracting Hours and Minutes:**


In [ ]:
# Function to extract hour and minute values from a time column
def extract_min_hour(df, col):
    df[col + "_hour"] = df[col].dt.hour  # Extracting the hour
    df[col + "_minute"] = df[col].dt.minute  # Extracting the minute

# Applying the function to extract hours and minutes from Dep_Time and Arrival_Time columns
extract_min_hour(dataset, "Dep_Time")
extract_min_hour(dataset, "Arrival_Time")


**Flight Duration Preprocessing:**


In [ ]:
# Function to convert duration strings to total minutes
def convert_duration_to_minutes(duration_str):
    """
    Convert a duration string in the format 'Xh Ym' to total minutes.
    Example: '2h 50m' -> 170
    """
    # Initialize total minutes
    total_minutes = 0
    
    # Split the string by spaces to extract hours and minutes
    parts = duration_str.split()
    
    for part in parts:
        if 'h' in part:
            # Extract hours and convert to minutes
            hours = int(part.replace('h', ''))
            total_minutes += hours * 60
        elif 'm' in part:
            # Extract minutes
            minutes = int(part.replace('m', ''))
            total_minutes += minutes
    
    return total_minutes

# Apply the function to the 'Duration' column
dataset['Duration_total_min'] = dataset['Duration'].apply(convert_duration_to_minutes)


## **3. Exploratory Data Analysis (EDA)**

**Departure Time Distribution:**


In [ ]:
# Function to categorize flights into different times of the day based on departure time
def flight_dep_time(x):
    if (x > 4) and (x <= 8):
        return "Early Morning"  # Flights departing between 4 AM and 8 AM
    elif (x > 8) and (x <= 12):
        return "Morning"  # Flights departing between 8 AM and 12 PM
    elif (x > 12) and (x <= 16):
        return "Afternoon"  # Flights departing between 12 PM and 4 PM
    elif (x > 16) and (x <= 20):
        return "Evening"  # Flights departing between 4 PM and 8 PM
    elif (x > 20) and (x <= 24):
        return "Night"  # Flights departing between 8 PM and midnight
    else:
        return "Late Night"  # Flights departing after midnight or before 4 AM

# Applying the function to categorize departure times in the dataset
dataset['Dep_Time_Category'] = dataset['Dep_Time_hour'].apply(flight_dep_time)

# Plotting the distribution of flight departure times across the different categories
dataset['Dep_Time_Category'].value_counts().plot(kind="bar", color='green')
plt.show()  # Display the plot




**Correlation Between Duration and Price:**


In [ ]:
# Visualizing the relationship between total flight duration (in minutes) and ticket price
# Color-coding based on the number of stops (hue='Total_Stops')
sns.scatterplot(x='Duration_total_min', y='Price', data=dataset, hue='Total_Stops')
plt.show()  # Display the scatter plot


In [ ]:
# Creating a boxplot to compare the price distributions across different airlines
# Sorting airlines by average price to improve readability
sns.boxplot(x='Airline', y='Price', data=dataset.sort_values("Price", ascending=False))
plt.xticks(rotation="vertical")  # Rotating x-axis labels to avoid overlap
plt.show()  # Display the boxplot


## **4. Feature Encoding**



**One-Hot Encoding for Categorical Features:**


In [ ]:
# One-hot encoding for the 'Source' column
# Create a new column for each unique value in the 'Source' column
# If the value matches the specific category, assign 1; otherwise, assign 0
for sub_category in dataset['Source'].unique():
    dataset['Source_' + sub_category] = dataset['Source'].apply(lambda x: 1 if x == sub_category else 0)


**Target Guided Encoding:**


In [ ]:
# Encoding the 'Airline' column based on the average price (target variable)
# Airlines with lower average prices get lower numerical ranks
airlines = dataset.groupby(['Airline'])['Price'].mean().sort_values().index
# Create a dictionary mapping airlines to their respective ranks
dict_airlines = {key: index for index, key in enumerate(airlines, 0)}
# Map the 'Airline' column to its corresponding rank based on the dictionary
dataset['Airline'] = dataset['Airline'].map(dict_airlines)


In [ ]:
# Check the unique values in the 'Total_Stops' column to understand the different stop options
# This will give us an idea of the categorical values we need to convert into numerical ones.
dataset['Total_Stops'].unique()

# Create a dictionary to map the 'Total_Stops' values (which are currently categorical) to numerical values.

stop = {'non-stop': 0, '2 stops': 2, '1 stop': 1, '3 stops': 3, '4 stops': 4}

# Replace the values in the 'Total_Stops' column using the dictionary.

dataset['Total_Stops'] = dataset['Total_Stops'].map(stop)

# The 'map' function applies the dictionary 'stop' to each entry in the 'Total_Stops' column.



In [ ]:
# Step 1: Group the dataset by 'Destination' and calculate the mean 'Price' for each group.
# This allows us to rank the destinations based on the average price for flights to that destination.
destinations = dataset.groupby(['Destination'])['Price'].mean().sort_values().index

# Step 2: Create a dictionary that maps each destination to a unique integer (starting from 0).
dict_destination = {index: key for key, index in enumerate(destinations, 0)}

# Step 3: Apply the encoding to the 'Destination' column in the dataset.
dataset['Destination'] = dataset['Destination'].map(dict_destination)


**Dropping the encoded columns:**

In [ ]:
def drop_encoded_columns(dataset, encoded_columns):
    """
    Drops the original columns that were encoded from the dataset.

    Parameters:
    - dataset: The DataFrame from which columns are to be dropped.
    - encoded_columns: A list of original columns that were encoded and need to be removed.

    Returns:
    - dataset: The DataFrame with the encoded columns removed.
    """
    # Drop the specified columns from the dataset
    dataset = dataset.drop(encoded_columns, axis=1)
    
    # Return the modified dataset with the encoded columns removed
    return dataset


In [ ]:
# List of original columns that were encoded
encoded_columns = ['Dep_Time', 'Arrival_Time', 'Date_of_Journey', 'Additional_Info', 'Source','Route','Duration','Dep_Time_Category']

# Call the function to drop these encoded columns
dataset = drop_encoded_columns(dataset, encoded_columns)


In [ ]:
dataset.head()

## **5. Outlier Detection and Treatment**



In [ ]:
# Function to plot the distribution, boxplot, and density of a column to check for outliers
def plot_distribution(col):
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))  # Create a 1x3 subplot
    sns.histplot(dataset[col], ax=axes[0])  # Histogram for the column
    sns.boxplot(x=dataset[col], ax=axes[1])  # Boxplot to detect outliers
    sns.kdeplot(dataset[col], ax=axes[2])  # Density plot for the distribution
    plt.show()  # Display all three plots

# Calling the function to check the distribution of 'Price'
plot_distribution('Price')

# Outlier treatment using the Interquartile Range (IQR) method
# First, calculate Q1 (25th percentile) and Q3 (75th percentile)
q1 = dataset['Price'].quantile(0.25)
q3 = dataset['Price'].quantile(0.75)

# Calculate the IQR (Q3 - Q1)
iqr = q3 - q1

# Define the acceptable range for outliers: [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
maximum = q3 + 1.5 * iqr
minimum = q1 - 1.5 * iqr

# Replace outliers in 'Price' (values greater than 'maximum') with the median price
dataset['Price'] = np.where(dataset['Price'] >= maximum, dataset['Price'].median(), dataset['Price'])


## **6. Feature Selection Using Mutual Information**



**Mutual Information for Feature Selection:**


In [ ]:
from sklearn.feature_selection import mutual_info_regression

# Defining the input features (X) and target variable (y)
X = dataset.drop(['Price'], axis=1)  # Drop 'Price' from the input features
y = dataset['Price']  # Target variable is 'Price'

# Calculate the mutual information between each input feature and the target variable
imp = mutual_info_regression(X, y)

# Create a DataFrame to store the importance values with the corresponding feature names
imp_df = pd.DataFrame(imp, index=X.columns, columns=['importance'])

# Sort features by importance in descending order
imp_df.sort_values(by='importance', ascending=False)

In [ ]:
dataset['Journey_Year']

In [ ]:
#Drop constant feature
dataset.drop('Journey_Year', axis=1, inplace=True)

Journey_Year feature is constant and can be dropped. 

## **7. Model Building**



**Splitting the data:**

In [ ]:
# Importing train_test_split to split the dataset into training and testing subsets
from sklearn.model_selection import train_test_split

# Splitting data: 75% training and 25% testing; random_state ensures the same data split each time
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


**Training regression models:**

In [ ]:
# Importing RandomForestRegressor for regression task
from sklearn.ensemble import RandomForestRegressor

# Creating a RandomForest model instance
ml_model = RandomForestRegressor()

# Fitting the RandomForest model using the training data
ml_model.fit(X_train, y_train)

# Making predictions on the test data
y_pred = ml_model.predict(X_test)

**Evaluating model performance:**

In [ ]:
# Importing metrics for model evaluation
from sklearn import metrics

# Calculating the R² score to evaluate the model's performance (how well the predictions match actual values)
metrics.r2_score(y_test, y_pred)

## **8. Model Saving & Deployment**

In [ ]:
# Importing pickle to save and load the trained model
import pickle

# Opening a file in write-binary mode to save the trained RandomForest model
file = open('/kaggle/working/randomforest_model.pkl', 'wb')

# Serializing and saving the trained model using pickle
pickle.dump(ml_model, file)

In [ ]:
# Loading the saved RandomForest model from the file
model = open('/kaggle/working/randomforest_model.pkl', 'rb')
forest = pickle.load(model)

# Using the loaded model to make predictions on the test set
y_pred2 = forest.predict(X_test)

# Evaluating the R² score again with the loaded model
metrics.r2_score(y_test, y_pred2)

In [ ]:
# Function to calculate Mean Absolute Percentage Error (MAPE)
def mape(y_true, y_pred):
    # Converting to NumPy arrays for easier manipulation
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    
    # Calculating the average absolute percentage error
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
# Function to train a given model, evaluate its performance, and plot the residuals
def predict(ml_model):
    # Fitting the model with training data
    model = ml_model.fit(X_train, y_train)
    
    # Printing the training score (R² on the training set)
    print('Training score: {}'.format(model.score(X_train, y_train)))
    
    # Predicting on the test data
    y_prediction = model.predict(X_test)
    
    # Calculating and printing the R² score on test data
    r2_score = metrics.r2_score(y_test, y_prediction)
    print('R² Score: {}'.format(r2_score))
    
    # Printing Mean Squared Error (MSE)
    print('MSE: {}'.format(metrics.mean_squared_error(y_test, y_prediction)))
    
    # Printing Mean Absolute Error (MAE)
    print('MAE: {}'.format(metrics.mean_absolute_error(y_test, y_prediction)))
    
    # Printing Mean Absolute Percentage Error (MAPE)
    print('MAPE: {}'.format(mape(y_test, y_prediction)))
    
    # Plotting a distribution of residuals (errors between true and predicted values)
    sns.displot(y_test - y_prediction)

In [ ]:
# Testing the predict function with a DecisionTreeRegressor model
from sklearn.tree import DecisionTreeRegressor
predict(DecisionTreeRegressor())

## **9. Hyperparameter Tuning**

In [ ]:
# Importing RandomizedSearchCV for hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# Defining a range of possible values for the hyperparameters of RandomForest
n_estimators = [int(x) for x in np.linspace(start=100, stop=1200, num=6)]
max_depth = [int(x) for x in np.linspace(start=5, stop=30, num=4)]
max_features = ['sqrt', 'auto']
min_samples_split = [5, 10, 15, 100]

# Creating a dictionary to hold hyperparameter options for the RandomForest model
dict_rf = {
    'n_estimators': n_estimators,
    'max_depth': max_depth,
    'max_features': max_features,
    'min_samples_split': min_samples_split
}

# Creating a RandomizedSearchCV instance to perform hyperparameter tuning
rf_rand = RandomizedSearchCV(estimator=RandomForestRegressor(), param_distributions=dict_rf, cv=3, n_jobs=-1, verbose=2)

# Fitting the RandomizedSearchCV on the training data to find the best combination of hyperparameters
rf_rand.fit(X_train, y_train)

# Accessing and printing the best hyperparameters found during the tuning process
rf_rand.best_params_

# Accessing and printing the best cross-validation score obtained
rf_rand.best_score_


In [ ]:
# Accessing and printing the best hyperparameters found during the tuning process
rf_rand.best_params_

In [ ]:
# Accessing and printing the best cross-validation score obtained
rf_rand.best_score_

## **10. Conclusion**

In this project, I developed and evaluated a Random Forest Regressor model for predicting flight prices based on a diverse set of features. 

I also explored several important regression metrics, including:

**R² score:** This provided insight into how well our model explained the variance in the data.

**Mean Squared Error (MSE) and Mean Absolute Error (MAE):** These helped us quantify the accuracy of the predictions.

**Mean Absolute Percentage Error (MAPE):** This gave us a percentage-based view of the model’s error, which is especially useful for understanding its performance relative to the true values.

Additionally, I employed **RandomizedSearchCV** to fine-tune hyperparameters of the Random Forest model, which allowed me to further optimize performance by adjusting parameters such as the number of estimators, tree depth, and feature selection method. This process improved both the efficiency and accuracy of our model.

The Random Forest and Decision Tree regressors were tested and compared. While both models provided valuable insights, the Random Forest demonstrated better generalization due to its ensemble nature, which helps reduce overfitting.

Furthermore, we implemented model persistence using **Pickle**, ensuring that the trained model can be easily saved and reloaded for future predictions without retraining. This is particularly useful for deployment scenarios.


